# Import packages :

In [1]:
import utils
from utils import *
import importlib
from torchvision.transforms import ToTensor

from sklearn.model_selection import train_test_split

from torch.utils.data import random_split

In [2]:
reset_val_dir()

# Dataset and Data Augmentation :

In [12]:
import os

all_filenames = [f for f in os.listdir("../dataset_project_iapr2025/train") if f.endswith(('.JPG'))]

train_filenames, val_filenames = train_test_split(all_filenames[13:], test_size=0.33, random_state=42)

train_filenames += all_filenames[:13]

In [13]:
# Path to your dataset folder

# Get all image filenames (JPG, JPEG, PNG, etc.)
# image_filenames = [f for f in os.listdir("../dataset_project_iapr2025/train") if f.endswith(('.JPG'))]

train_dataset = YoloDetectionAugmentedDataset(
	image_filenames=train_filenames,
	image_dir="../dataset_project_iapr2025/train",
	label_dir="./labels",
	anchors=ANCHORS,
	transform=YoloCompatibleTransform(train=True),
	num_augmentations=3
)

val_dataset = YoloDetectionAugmentedDataset(
	image_filenames=val_filenames,
	image_dir="../dataset_project_iapr2025/train",
	label_dir="./labels",
	anchors=ANCHORS,
	transform=YoloCompatibleTransform(train=False),
	num_augmentations=0
)

def yolo_collate_fn(batch):
	images, targets = zip(*batch)  # Unzips list of tuples
	images = torch.stack(images)
	targets = torch.stack(targets)
	return images, targets

train_loader = torch.utils.data.DataLoader(
	train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=yolo_collate_fn)

val_loader = torch.utils.data.DataLoader(
	val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=yolo_collate_fn)

Preprocessing images with fixed augmentations...


100%|██████████| 73/73 [00:59<00:00,  1.22it/s]


Preprocessing images with fixed augmentations...


100%|██████████| 30/30 [00:10<00:00,  2.99it/s]


In [5]:
num_images = len(train_loader.dataset)
print(f'Total number of images: {num_images}')
# for batch_size, (image, target) in enumerate(dataloader):
# 	# print(image)
# 	print(target)

Total number of images: 292


In [6]:
# images, targets = next(iter(train_loader))

# visualize_dataloader_all(
#     loader=train_loader,
#     # model=model,  # or None if just ground truth
#     device=DEVICE,
#     save_dir="train_visuals",
#     class_names=CLASS_NAMES,
#     anchors=SCALED_ANCHORS,
#     resized_img_width=RESIZED_IMG_WIDTH,
#     resized_img_height=RESIZED_IMG_HEIGHT,
#     grid_width=GRID_WIDTH,
#     grid_height=GRID_HEIGHT
# )

# Csv file generation

In [14]:
# ============ TEST SETUP FOR SUBMISSION ============ #


test_dir = "../dataset_project_iapr2025/test"
test_filenames = sorted([f for f in os.listdir(test_dir) if f.endswith(".JPG")])

test_dataset = YOLOTestDataset(
	filenames=test_filenames,
	img_dir="../dataset_project_iapr2025/test"
)

test_loader = torch.utils.data.DataLoader(
	test_dataset,
	batch_size=BATCH_SIZE,
	shuffle=False,
	pin_memory=True
)



# Functions training and evaluation :

# Training :

In [ ]:
importlib.reload(utils)
from utils import *
# Creating the model from YOLOv3 class 
model = EfficientYOLO_12x18Grid().to(DEVICE) 
model = init_yolo_weights(model)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {num_params}")

# Defining the optimizer 
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE) 

# Defining the loss function 
loss_fn = YOLOLoss() 


if LOAD_MODEL:
	load_checkpoint(CHECKPOINT_FILE, model, optimizer, LEARNING_RATE)
	
# Training the model
for e in range(1, NB_EPOCHS+1): 
	print("Epoch:", e) 
	training_loop(train_loader, model, optimizer, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE) 

	# Validation
	val_loss = evaluate(val_loader, model, loss_fn, scaled_anchors=SCALED_ANCHORS, device=DEVICE)
	print(f"Validation Loss: {val_loss:.4f}")
	
	# Save validation images
	if e % 10 == 0:
		save_val_images(model, val_loader, device=DEVICE, scaled_anchors=SCALED_ANCHORS, save_dir="val_images", epoch=e, class_names = CLASS_NAMES, grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT, resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT)

	# Saving the model 
	if SAVE_MODEL: 
		save_checkpoint(model, optimizer, filename=f"checkpoint.pth.tar")

# Generating the submission CSV file
generate_submission_csv_with_nms(
	model, 
	test_loader, 
	output_csv_path="output/test_submission.csv"
)


Nombre de paramètres entraînables : 8799362
Epoch: 1


100%|██████████| 10/10 [01:33<00:00,  9.31s/it, loss=3.67]


Validation Loss: 3.0973
==> Saving checkpoint
Epoch: 2


 30%|███       | 3/10 [00:30<01:11, 10.18s/it, loss=3.33]

# Check the transformations :

In [ ]:
def save_gt_images(loader, class_names=CLASS_NAMES, save_dir="gt_visualizations", grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT):
	"""
	Saves images with ground truth bounding boxes from a DataLoader
	
	Args:
		loader: PyTorch DataLoader (train or val)
		class_names: List of class names
		save_dir: Directory to save visualized images
	"""
	os.makedirs(save_dir, exist_ok=True)
	print(f"Saving ground truth images to {save_dir}/")
	# Color map for different classes
	colour_map = plt.get_cmap("tab20b")
	colors = [colour_map(i) for i in np.linspace(0, 1, len(class_names))]
	
	for batch_idx, (images, targets) in enumerate(tqdm(loader)):
		# Convert images from tensor to numpy and denormalize
		images = images.permute(0, 2, 3, 1).cpu().numpy() # shape (batch_size, channels, height, width) to (batch_size, height, width, channels)
		images = (images * 255).astype(np.uint8)
		
		# Process each image in the batch
		for img_idx in range(images.shape[0]):
			img = images[img_idx]
			fig, ax = plt.subplots(1, figsize=(12, 8))
			ax.imshow(img)
			
			# Get boxes from the unique scale
			all_boxes = []
			target = targets[img_idx]
			anchor_indices = torch.where(target[..., 0] == 1)  # Cells with objects
			
			for anchor, i, j in zip(*anchor_indices):
				# for ex anchor = 0, i = 7, j = 15
				# Get box coordinates : x,y in [0,1] relative to the cell, w,h in [0,grid_width-1] and [0, grid_height-1] so relative to the grid size
				x, y, w, h = target[anchor, i, j, 1:5]
				class_id = target[anchor, i, j, 5].int() # in [0, 12]
				
				
				# Calculate box coordinates normalized to [0,1] with respect to the resized image size
				x_center = (j + x) / grid_width # normalize to [0,1]
				y_center = (i + y) / grid_height
				width = w / grid_width
				height = h / grid_height

				all_boxes.append([
					class_id.item(),  # class
					1.0,             # confidence (ground truth so 1)
					x_center.item(),
					y_center.item(),
					width.item(),
					height.item()
				])
			
			print(len(all_boxes))
			
			# Draw all ground truth boxes
			for box in all_boxes:
				class_id, conf, x, y, w, h = box
				
				# Calculate coordinates
				x1 = (x - w/2) * img.shape[1] # finds the upper-left corner of the box in pixels
				y1 = (y - h/2) * img.shape[0]
				width = w * img.shape[1]
				height = h * img.shape[0]
				
				# Create rectangle patch
				rect = patches.Rectangle(
					(x1, y1), width, height,
					linewidth=2,
					edgecolor=colors[int(class_id)],
					facecolor="none"
				)
				ax.add_patch(rect)
				
				# Add label
				plt.text(
					x1, y1,
					s=f"{class_names[int(class_id)]} {conf:.2f}",
					color="white",
					verticalalignment="top",
					bbox={"color": colors[int(class_id)], "pad": 0},
				)
			
			# Save image
			plt.axis("off")
			plt.savefig(
				os.path.join(save_dir, f"img_{batch_idx + img_idx}.png"),
				bbox_inches="tight",
				pad_inches=0,
				dpi=100
			)
			plt.close()

# Usage example
save_gt_images(
   loader=val_loader,
   class_names=CLASS_NAMES,
   save_dir="train_gt_visualizations"
)
print(f"Saved ground truth visualizations to train_gt_visualizations/")


In [ ]:
len(train_loader)